
# YAMNet Gunshot Detection (OFFLINE + Network-Mount Friendly)

This notebook scans large `.WAV/.wav` files for **gunshot-like events** using **YAMNet only**.

### Why this version
Your lab audio is on a mounted drive (`/Volumes/...`). Loading entire WAVs via `librosa.load` can:
- fall back to `audioread` (slow)
- time out on network mounts

So this notebook:
- **Optionally copies each WAV to a local cache** before processing (recommended)
- **Processes audio in streaming blocks** (no full-file load)

### Outputs
- Top timestamps per file where YAMNet gunshot probability peaks
- Spectrogram PNG + short audio clip for each detection


In [ ]:

# ========================
# USER SETTINGS (EDIT ME)
# ========================

from pathlib import Path

# Folder containing WAVs for a given date (recursive)
LAB_ROOT = Path(
    "/Volumes/aid_elephants_interaction/Audio Data/2025/01_FR46/05_05_25 to 05_20_25"
)

# Start small
MAX_LAB_FILES = 10

# Detection tuning
THRESHOLD = 0.30          # raise to reduce false positives
MIN_SEPARATION_SEC = 2.0  # minimum spacing between detections
TOP_K_PER_FILE = 5

# Streaming settings
BLOCK_SEC = 120.0          # process audio in 60s chunks
TARGET_SR = 16000         # YAMNet expects 16kHz

# Network-mount robustness
USE_LOCAL_CACHE = True
CACHE_DIR = Path("wav_cache")   # local folder on your machine
CACHE_DIR.mkdir(exist_ok=True)

# Output directory
OUT_DIR = Path("gunshot_outputs")
OUT_DIR.mkdir(exist_ok=True)

print("LAB_ROOT:", LAB_ROOT)
print("USE_LOCAL_CACHE:", USE_LOCAL_CACHE)


LAB_ROOT: /Volumes/aid_elephants_interaction/Audio Data/2025/01_FR46/05_05_25 to 05_20_25
USE_LOCAL_CACHE: True


In [24]:

# ========================
# IMPORTS & MODEL LOADING
# ========================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
import shutil
from tqdm import tqdm
import tensorflow_hub as hub
import tensorflow as tf

yamnet = hub.load("https://tfhub.dev/google/yamnet/1")

# Hard-coded gunshot index for YAMNet v1
GUNSHOT_CLASS_INDEX = 418
print("Using GUNSHOT_CLASS_INDEX =", GUNSHOT_CLASS_INDEX)


Using GUNSHOT_CLASS_INDEX = 418


In [25]:

# ========================
# HELPERS: file discovery
# ========================

from pathlib import Path

def find_day_folders(root: Path):
    """
    Return subfolders that look like YYYYMMDD
    """
    days = []
    for p in root.iterdir():
        if p.is_dir() and p.name.isdigit() and len(p.name) == 8:
            days.append(p)
    return sorted(days)

def find_wavs_in_day(day_folder: Path):
    return sorted([
        p for p in day_folder.rglob("*")
        if p.is_file() and p.suffix.lower() == ".wav"
    ])

day_folders = find_day_folders(LAB_ROOT)

print("Found day folders:")
for d in day_folders:
    print(" -", d.name)

print("Total day folders:", len(day_folders))



Found day folders:
 - 19700101
 - 20250505
 - 20250506
 - 20250507
 - 20250508
 - 20250509
 - 20250510
 - 20250511
 - 20250512
 - 20250513
Total day folders: 10


In [26]:

# ========================
# HELPERS: local caching
# ========================

def local_cache_path(src: Path, cache_dir: Path) -> Path:
    # keep unique filename; if collisions are possible, include parent hash
    return cache_dir / src.name

def ensure_local_copy(src: Path, cache_dir: Path, retries=2, sleep_sec=5) -> Path | None:
    """
    Try to copy src -> cache. If it fails (mount hiccup), return None so we can skip this file.
    """
    dst = local_cache_path(src, cache_dir)

    # Already cached and sizes match
    try:
        if dst.exists() and dst.stat().st_size == src.stat().st_size:
            return dst
    except Exception:
        pass

    for attempt in range(1, retries + 1):
        try:
            shutil.copy2(src, dst)
            return dst
        except Exception as e:
            print(f"[cache] Copy failed (attempt {attempt}/{retries}) for {src.name}: {type(e).__name__}: {e}")
            time.sleep(sleep_sec)

    print(f"[cache] SKIPPING (unreadable right now): {src}")
    return None


In [27]:

# ========================
# HELPERS: streaming read + resample
# ========================

def to_mono(x: np.ndarray) -> np.ndarray:
    if x.ndim == 1:
        return x
    # x shape: (n, channels)
    return np.mean(x, axis=1)

def resample_to_16k(y: np.ndarray, orig_sr: int, target_sr: int = TARGET_SR) -> np.ndarray:
    if orig_sr == target_sr:
        return y.astype(np.float32)
    return librosa.resample(y.astype(np.float32), orig_sr=orig_sr, target_sr=target_sr)

def yamnet_scores_for_block(y_16k: np.ndarray):
    scores, _, _ = yamnet(y_16k.astype(np.float32))
    return scores.numpy()

def iter_audio_blocks(path: Path, block_sec: float):
    """Yield (block_audio, sr, start_time_sec) from a WAV without loading full file."""
    with sf.SoundFile(str(path)) as f:
        sr = f.samplerate
        channels = f.channels
        block_frames = int(block_sec * sr)

        idx = 0
        while True:
            data = f.read(block_frames, dtype="float32", always_2d=(channels > 1))
            if data is None or len(data) == 0:
                break
            # data: (frames, channels) if always_2d else (frames,)
            if data.ndim == 2:
                data = to_mono(data)
            start_time = idx * block_sec
            yield data, sr, start_time
            idx += 1


In [28]:

# ========================
# DETECTION: peak picking
# ========================

def peak_pick_times(times, scores, top_k=5, min_separation_sec=1.0):
    order = np.argsort(scores)[::-1]
    chosen = []
    for i in order:
        t = float(times[i])
        if all(abs(t - ct) >= min_separation_sec for ct in chosen):
            chosen.append(t)
        if len(chosen) >= top_k:
            break
    return chosen


In [29]:

# ========================
# SCAN ONE FILE (streaming)
# ========================

def scan_file_streaming(
    wav_path: Path,
    gunshot_class_index: int,
    threshold: float,
    top_k: int,
    min_separation_sec: float,
    block_sec: float,
    use_local_cache: bool,
    cache_dir: Path
):
    src_path = Path(wav_path)

    # Local cache to avoid /Volumes timeouts
    path = src_path
    if use_local_cache:
        cached = ensure_local_copy(src_path, cache_dir)
        if cached is None:
            return pd.DataFrame(columns=["file","time_sec","gunshot_prob"])
        path = cached


    # We’ll collect per-frame gunshot probs & corresponding absolute times
    all_probs = []
    all_times = []

    # YAMNet frame hop is about 0.48s
    hop_sec = 0.48

    for block, sr, block_start in iter_audio_blocks(path, block_sec=block_sec):
        y16 = resample_to_16k(block, orig_sr=sr, target_sr=TARGET_SR)
        scores = yamnet_scores_for_block(y16)
        probs = scores[:, gunshot_class_index]

        times = block_start + (np.arange(len(probs)) * hop_sec)

        all_probs.append(probs)
        all_times.append(times)

    if len(all_probs) == 0:
        return pd.DataFrame(columns=["file","time_sec","gunshot_prob"])

    probs = np.concatenate(all_probs)
    times = np.concatenate(all_times)

    # threshold filter
    if np.max(probs) < threshold:
        return pd.DataFrame(columns=["file","time_sec","gunshot_prob"])

    top_times = peak_pick_times(times, probs, top_k=top_k, min_separation_sec=min_separation_sec)

    rows = []
    for t in top_times:
        # find nearest frame
        i = int(np.argmin(np.abs(times - t)))
        if probs[i] >= threshold:
            rows.append({"file": str(src_path), "time_sec": float(times[i]), "gunshot_prob": float(probs[i])})

    return pd.DataFrame(rows).sort_values("gunshot_prob", ascending=False)


In [30]:

# ========================
# SAVE SPECTROGRAM + CLIP
# ========================

def load_snippet(path: Path, center_time_sec: float, segment_sec: float = 3.0):
    # read snippet directly with soundfile for speed/reliability
    with sf.SoundFile(str(path)) as f:
        sr = f.samplerate
        start_frame = max(0, int((center_time_sec - segment_sec/2) * sr))
        n_frames = int(segment_sec * sr)
        f.seek(start_frame)
        data = f.read(n_frames, dtype="float32", always_2d=(f.channels > 1))
        if data.ndim == 2:
            data = np.mean(data, axis=1)
    return data, sr

def save_segment_spectrogram(src_path: Path, center_time_sec: float, out_png: Path, segment_sec: float = 3.0):
    # use cached file if enabled
    path = ensure_local_copy(src_path, CACHE_DIR) if USE_LOCAL_CACHE else src_path

    y, sr = load_snippet(path, center_time_sec, segment_sec=segment_sec)
    y16 = resample_to_16k(y, orig_sr=sr, target_sr=TARGET_SR)

    S = librosa.stft(y16, n_fft=1024, hop_length=256)
    S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)

    plt.figure(figsize=(10, 4))
    librosa.display.specshow(S_db, sr=TARGET_SR, hop_length=256, x_axis="time", y_axis="hz")
    plt.colorbar(format="%+2.0f dB")
    plt.title(f"{src_path.name} | t={center_time_sec:.2f}s")
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()

def save_audio_clip(src_path: Path, center_time_sec: float, out_wav: Path, segment_sec: float = 3.0):
    path = ensure_local_copy(src_path, CACHE_DIR) if USE_LOCAL_CACHE else src_path
    y, sr = load_snippet(path, center_time_sec, segment_sec=segment_sec)
    # save at original sr for listening
    sf.write(str(out_wav), y, sr)


In [31]:
all_hits = []

for day in day_folders:
    print(f"\n=== Processing day {day.name} ===")

    wavs = find_wavs_in_day(day)
    print(f"  WAV files found: {len(wavs)}")

    if len(wavs) == 0:
        continue

    # optional: limit per day to avoid overload
    wavs = wavs[:MAX_LAB_FILES]

    for wav in tqdm(wavs, desc=f"Scanning {day.name}", leave=False):
        hits = scan_file_streaming(
            wav_path=wav,
            gunshot_class_index=GUNSHOT_CLASS_INDEX,
            threshold=THRESHOLD,
            top_k=TOP_K_PER_FILE,
            min_separation_sec=MIN_SEPARATION_SEC,
            block_sec=BLOCK_SEC,
            use_local_cache=USE_LOCAL_CACHE,
            cache_dir=CACHE_DIR
        )

        # Save artifacts
        for r in hits.itertuples(index=False):
            base = Path(r.file).stem
            t = r.time_sec
            spec_path = OUT_DIR / f"{day.name}_{base}_t{t:.2f}_spec.png"
            clip_path = OUT_DIR / f"{day.name}_{base}_t{t:.2f}_clip.wav"

            save_segment_spectrogram(Path(r.file), t, spec_path)
            save_audio_clip(Path(r.file), t, clip_path)

        if len(hits) > 0:
            hits = hits.copy()
            hits["day"] = day.name
            all_hits.append(hits)

# Final combined table
results = (
    pd.concat(all_hits, ignore_index=True)
    if all_hits
    else pd.DataFrame(columns=["file", "day", "time_sec", "gunshot_prob"])
)

results.sort_values("gunshot_prob", ascending=False).head(30)



=== Processing day 19700101 ===
  WAV files found: 1



=== Processing day 20250505 ===
  WAV files found: 5



=== Processing day 20250506 ===
  WAV files found: 10



=== Processing day 20250507 ===
  WAV files found: 10



=== Processing day 20250508 ===
  WAV files found: 10



=== Processing day 20250509 ===
  WAV files found: 10



=== Processing day 20250510 ===
  WAV files found: 10



=== Processing day 20250511 ===
  WAV files found: 10


[cache] Copy failed (attempt 1/2) for 247AA50763B3E006_20250511_190000.WAV: OSError: [Errno 28] No space left on device: '/Volumes/aid_elephants_interaction/Audio Data/2025/01_FR46/05_05_25 to 05_20_25/20250511/247AA50763B3E006_20250511_190000.WAV' -> 'wav_cache/247AA50763B3E006_20250511_190000.WAV'


NameError: name 'time' is not defined


## If you still see timeouts
- Keep `USE_LOCAL_CACHE = True` (recommended)
- Increase `BLOCK_SEC` to 120 (fewer reads)
- Or copy the entire day folder locally once, then point `LAB_ROOT` to the local folder
